# Corruption-Validated Judge Audit for the Policy Q&A Layer

No automated judge is trusted to grade this paper's policy layer until its error rates
are measured on answers whose defects are known by construction. This notebook runs that
validation and rebuilds every number reported in manuscript Sections 3.8 and 4.7.

- **Items**: the 25 executed demo answers (from `rag_pipeline.ipynb`) plus 4 corruption
  types x 25 copies = 125 items, judged blind.
- **Judges** (generator `gemma4:e4b` excluded to avoid self-preference):
  `qwen3.5:9b`, `gpt-oss:20b` (different families), `llama3.1:8b` (a commonly
  adopted small judge, included as a control).
- **Selection gates**: detection >= 0.80 on corrupted items AND flag rate <= 0.20 on
  uncorrupted items.

The run is checkpointed to `judge_audit_checkpoint.csv`: verdicts already present are
reused and only missing (judge, item) pairs call Ollama (~30 s per call, ~2 h for the
full 375-call battery). With the shipped checkpoint the notebook executes in seconds and
makes zero LLM calls.

## 1. Setup — extract the demo answers and build the corruption battery

In [1]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

np.random.seed(42)

JUDGES = ['qwen3.5:9b', 'gpt-oss:20b', 'llama3.1:8b']
CKPT = Path('judge_audit_checkpoint.csv')
TYPES = ['citation_swap', 'fabricated_statistic', 'fabricated_reference',
         'recommendation_overreach']
TYPE_PREFIX = {'citation_swap': 'cita', 'fabricated_statistic': 'fsta',
               'fabricated_reference': 'fref', 'recommendation_overreach': 'reco'}

WHO_DOCS = [
    "GLASS Report 2021 / 2022 (Global antimicrobial resistance and use surveillance system)",
    "WHO Global Action Plan on Antimicrobial Resistance (2015)",
    "GLASS Manual for antimicrobial resistance surveillance",
    "WHO Bacterial Priority Pathogens List 2024",
    "WHO Policy Guidance on Integrated Antimicrobial Stewardship Activities",
]
TRUE_METRICS = (
    "Reference forecast metrics (audited panel, 2023 test): best method "
    "Hybrid selector MAE 5.74 pp; Persistence 5.79; XGBoost 6.55; "
    "LightGBM 6.53; Linear 6.39; naive global mean 24.68; "
    "regional MAE: Europe 3.03, Western Pacific 4.61, Africa 7.64, "
    "Eastern Mediterranean 7.82, South-East Asia 7.91; regional ratio 2.6x; "
    "top feature: prior-year resistance 54.0%."
)

def load_qa():
    nb = json.load(open('rag_pipeline.ipynb', encoding='utf-8'))
    full = ''
    for c in nb['cells']:
        for o in c.get('outputs', []):
            t = o.get('text', '')
            full += ''.join(t) if isinstance(t, list) else t
    parts = re.split(r'={60,}', full)
    qa = []
    for i in range(len(parts) - 1):
        q = parts[i].strip().splitlines()[-1] if parts[i].strip() else ''
        if q.startswith('? '):
            ans = parts[i + 1].strip()
            if ans:
                qa.append((q[2:].strip(), ans))
    return qa

def corrupt(answer, ctype, idx):
    if ctype == 'citation_swap':
        def shift(m):
            k = int(m.group(1))
            return f'[Source {k % 5 + 1}]'
        out = re.sub(r'\[Source (\d+)\]', shift, answer)
        out = out.replace('[AMR FORECASTING RESULTS]', '[Source 3]')
        out = out.replace('[FORECASTING DATA]', '[Source 4]')
        if out == answer:
            sents = out.rstrip().rsplit('. ', 1)
            out = (sents[0] + '. [Source 4]' +
                   (' ' + sents[1] if len(sents) > 1 else ''))
        return out
    if ctype == 'fabricated_statistic':
        return (answer.rstrip() + ' Additionally, the forecasting system reports '
                'that the hybrid model achieves a test MAE of 2.1 percentage '
                'points uniformly across all WHO regions [AMR FORECASTING RESULTS].')
    if ctype == 'fabricated_reference':
        return (answer.rstrip() + ' This guidance is further elaborated in the '
                'WHO GLASS Technical Implementation Report 2027 '
                '(WHO/GLASS/TR/2027.4), which extends these surveillance '
                'targets to 2030.')
    if ctype == 'recommendation_overreach':
        return (answer.rstrip() + ' On this basis, national ministries should '
                'immediately restrict carbapenem prescribing nationwide and '
                'suspend empiric use in all outpatient settings without '
                'further review.')
    raise ValueError(ctype)

qa = load_qa()
assert len(qa) == 25, f'expected 25 Q&A pairs, got {len(qa)}'
items = []
for i, (q, a) in enumerate(qa):
    items.append({'item_id': f'c{i:02d}', 'condition': 'clean', 'question': q, 'answer': a})
    for t in TYPES:
        items.append({'item_id': f'{TYPE_PREFIX[t]}{i:02d}', 'condition': t,
                      'question': q, 'answer': corrupt(a, t, i)})
ids = [it['item_id'] for it in items]
assert len(ids) == len(set(ids))
print(f'{len(qa)} answers -> {len(items)} audit items')

25 answers -> 125 audit items


## 2. Judge prompt and single-call runner

In [2]:
import ollama

SYSTEM = f"""You are auditing ONE policy Q&A answer produced by a retrieval-augmented system whose knowledge base is exactly these WHO documents:
{chr(10).join('- ' + d for d in WHO_DOCS)}

{TRUE_METRICS}

Citations in answers appear as [Source N] (N = 1..5, retrieved WHO passages), [AMR FORECASTING RESULTS], or [FORECASTING DATA] (the reference metrics above).

Flag the answer CORRUPT if it contains any of these integrity problems:
1. citation_swap: a claim is tagged with the wrong source number or wrong provenance type.
2. fabricated_statistic: a numeric claim that contradicts, or is absent from, the reference metrics or the listed WHO documents.
3. fabricated_reference: a citation to a document that is not one of the five listed above.
4. recommendation_overreach: a strong directive recommendation (restrict, suspend, mandate) not supported by the cited evidence.

Otherwise verdict CLEAN. Reply with JSON only, no other text:
{{"verdict": "CORRUPT" or "CLEAN", "type": "citation_swap|fabricated_statistic|fabricated_reference|recommendation_overreach|none", "reason": "<=25 words"}}"""

def strip_think(text):
    text = re.sub(r'<think>.*?</think>', '', text, flags=re.S)
    text = re.sub(r'<\|channel_\|>analysis.*?<\|message\|>', '', text, flags=re.S)
    text = re.sub(r'<\|start\|>assistant<\|channel\|>final<\|message\|>', '', text)
    return text.strip()

def parse_verdict(raw):
    raw = strip_think(raw)
    for obj in reversed(re.findall(r'\{[^{}]*\}', raw, re.S)):
        try:
            d = json.loads(obj)
            v = str(d.get('verdict', '')).upper()
            t = str(d.get('type', 'none')).lower()
            if v in ('CORRUPT', 'CLEAN'):
                return {'verdict': v, 'type': 'none' if v == 'CLEAN' else t}
        except json.JSONDecodeError:
            continue
    return None

def judge_one(model, question, answer):
    msgs = [{'role': 'system', 'content': SYSTEM},
            {'role': 'user',
             'content': f'Question: {question}\n\nAnswer to audit:\n{answer}'}]
    npred = 2000 if model.startswith('gpt-oss') else 700
    opts = {'temperature': 0, 'num_predict': npred, 'seed': 42}
    try:
        r = ollama.chat(model=model, messages=msgs, options=opts, think=False)
    except TypeError:
        r = ollama.chat(model=model, messages=msgs, options=opts)
    msg = r['message']
    raw = msg.get('content') or ''
    if not raw.strip():
        raw = msg.get('thinking') or ''
    return parse_verdict(raw), raw

print('judge runner ready (Ollama must be running for uncached calls)')

judge runner ready (Ollama must be running for uncached calls)


## 3. Run / resume the audit
Existing checkpoint entries are reused; only missing (judge, item) pairs call Ollama.
With the shipped checkpoint this cell performs zero LLM calls.

In [3]:
done = set()
if CKPT.exists():
    prev = pd.read_csv(CKPT)
    done = {(r.judge, r.item_id) for r in prev.itertuples() if pd.notna(r.verdict)}
todo = [(j, it) for j in JUDGES for it in items if (j, it['item_id']) not in done]
print(f'{len(items)} items x {len(JUDGES)} judges = {len(items)*len(JUDGES)} calls; '
      f'{len(done)} cached, {len(todo)} to run')

rows = []
for n, (j, it) in enumerate(todo, 1):
    try:
        v, raw = judge_one(j, it['question'], it['answer'])
    except Exception as e:
        v, raw = None, f'ERROR: {e}'
    rows.append({'judge': j, 'item_id': it['item_id'], 'condition': it['condition'],
                 'verdict': v['verdict'] if v else 'PARSE_FAIL',
                 'type': v['type'] if v else '', 'raw': raw[:300]})
    if n % 10 == 0 or n == len(todo):
        new = pd.DataFrame(rows)
        allr = pd.concat([pd.read_csv(CKPT), new]) if CKPT.exists() else new
        allr = allr.drop_duplicates(subset=['judge', 'item_id'], keep='last')
        allr.to_csv(CKPT, index=False)
        print(f'[{n}/{len(todo)}] {j} {it["item_id"]} -> {rows[-1]["verdict"]}', flush=True)
print('audit complete ->', CKPT)

125 items x 3 judges = 375 calls; 375 cached, 0 to run
audit complete -> judge_audit_checkpoint.csv


## 4. Validation report (manuscript Table: judge validation)

In [4]:
def gwet_ac1(a, b):
    n = len(a)
    cats = sorted(set(a) | set(b))
    pa = np.mean([x == y for x, y in zip(a, b)])
    pk = [(a.count(c) + b.count(c)) / (2 * n) for c in cats]
    pe = sum(p * (1 - p) for p in pk) / (len(cats) - 1) if len(cats) > 1 else 1.0
    return (pa - pe) / (1 - pe) if pe < 1 else float('nan')

df = pd.read_csv(CKPT).drop_duplicates(subset=['judge', 'item_id'], keep='last')
print('parse failures by judge:',
      df[df.verdict == 'PARSE_FAIL'].groupby('judge').size().to_dict())
df = df[df.verdict != 'PARSE_FAIL']

summary = []
for j in df.judge.unique():
    d = df[df.judge == j]
    det = d[d.condition != 'clean'].verdict.eq('CORRUPT')
    fp = d[d.condition == 'clean'].verdict.eq('CORRUPT')
    row = {'judge': j, 'detection': det.mean(), 'clean_FP': fp.mean()}
    for t in TYPES:
        row[t] = d[d.condition == t].verdict.eq('CORRUPT').mean()
    typed = d[(d.condition != 'clean') & (d.verdict == 'CORRUPT')].copy()
    typed['type'] = typed['type'].fillna('')
    row['type_correct'] = (typed.type.values == typed.condition.values).mean() if len(typed) else np.nan
    summary.append(row)
sm = pd.DataFrame(summary).set_index('judge')
print()
print(sm.round(2).to_string())
print()
print('SELECTED (detection >= 0.80 and clean_FP <= 0.20):',
      sm[(sm.detection >= 0.80) & (sm.clean_FP <= 0.20)].index.tolist())

piv = df.pivot_table(index='item_id', columns='judge', values='verdict',
                     aggfunc='first').dropna()
print()
print('pairwise agreement (Gwet AC1 / raw):')
js = list(piv.columns)
for a in range(len(js)):
    for b in range(a + 1, len(js)):
        la, lb = list(piv[js[a]]), list(piv[js[b]])
        print(f'  {js[a]} vs {js[b]}: AC1={gwet_ac1(la, lb):.3f} '
              f'agree={np.mean([x == y for x, y in zip(la, lb)]):.3f}')
sm.to_csv('judge_validation_summary.csv', index=True)

parse failures by judge: {}

             detection  clean_FP  citation_swap  fabricated_statistic  fabricated_reference  recommendation_overreach  type_correct
judge                                                                                                                              
qwen3.5:9b        0.83      0.16           0.32                   1.0                   1.0                       1.0          0.98
gpt-oss:20b       0.97      0.64           0.88                   1.0                   1.0                       1.0          0.90
llama3.1:8b       0.76      0.04           0.04                   1.0                   1.0                       1.0          0.71

SELECTED (detection >= 0.80 and clean_FP <= 0.20): ['qwen3.5:9b']

pairwise agreement (Gwet AC1 / raw):
  gpt-oss:20b vs llama3.1:8b: AC1=0.521 agree=0.696
  gpt-oss:20b vs qwen3.5:9b: AC1=0.647 agree=0.760
  llama3.1:8b vs qwen3.5:9b: AC1=0.796 agree=0.888


## 5. Verdicts on the 25 real answers (validated judge)

In [5]:
sel = sm[(sm.detection >= 0.80) & (sm.clean_FP <= 0.20)].index.tolist()
judge = sel[0]
d = df[(df.judge == judge) & (df.condition == 'clean')]
print(f'validated judge: {judge}')
print(f'clean verdicts: {(d.verdict == "CLEAN").sum()} of {len(d)}')
flagged = d[d.verdict == 'CORRUPT']
for _, r in flagged.iterrows():
    print(f'  FLAG {r.item_id} [{r.type}]: {str(r.raw)[:160]}')

validated judge: qwen3.5:9b
clean verdicts: 21 of 25
  FLAG c01 [citation_swap]: {"verdict": "CORRUPT", "type": "citation_swap", "reason": "\"Specific GLASS target pathogens\" claim incorrectly cites Source 2 instead of the correct source fo
  FLAG c12 [citation_swap]: {"verdict": "CORRUPT", "type": "citation_swap", "reason": "\"2.6 times\" ratio refers to regional disparity, not high-income vs mature systems (GLASS Report)."}
  FLAG c17 [citation_swap]: {"verdict": "CORRUPT", "type": "citation_swap", "reason": "Specific antibiotics monitored are incorrectly attributed to Source 1 instead of the correct source."
  FLAG c24 [fabricated_statistic]: {"verdict": "CORRUPT", "type": "fabricated_statistic", "reason": "Answer claims Tricycle project is specifically related to ESBL E. coli, which contradicts WHO 
